In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tifffile
from ipywidgets import interact, widgets, fixed

from scipy.ndimage import gaussian_filter

# Add src to path if running from notebooks folder
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
from scene import *
    
%matplotlib inline

# 1) Synthetic cell generation

## 1.1) Cell shape gen

In [ ]:
# implement cells as star shaped polygons.
def generate_single_cell(size = 201, radius = 32, nuc_frac = 0.3, rough = 0.9, K = 20, elong = 1.6, angle_deg = 30, beta = 1.9):
    # Random fourrier coefficients for a star shaped polygon
    rng = np.random.default_rng(
        # seed=42
        )
    a, b = rng.standard_normal(K), rng.standard_normal(K)
    k = np.arange(1, K + 1)
    
    # get polar coordinates of the star shaped polygon
    # generate coordinate grid with origin in the center
    y, x = np.mgrid[:size, :size] - (size-1)/2
    t = np.deg2rad(angle_deg)
    # rotate the coordinate grid by angle_deg
    xr, yr = x * np.cos(t) + y * np.sin(t), -x * np.sin(t) + y * np.cos(t)
    # elongation factor along the x-axis
    s = np.sqrt(elong)
    # get polar coordinates
    rho = np.hypot(xr/s, yr*s)
    phi = np.arctan2(yr*s, xr/s)
    
    phi_q = np.linspace(0, 2*np.pi, 1024)          # dense angle grid, define once

    def boundary(R, kappa, extra = 0.0):
        # boundry radius as function of angle adding wobbles
        # get more fat lobes 
        w = kappa* k ** -(beta + extra)
        # decouple the scale from the amount of wobble
        w = w / (np.linalg.norm(w) + 1e-12) * kappa
        
        def g(p):
            # generate lumps at random locations (cosine and sine components)   
            return np.cos(p[..., None]*k) @ (w*a) + np.sin(p[..., None]*k) @ (w*b)
        
        area = 0.5 * np.trapezoid(np.exp(g(phi_q))**2, phi_q)   # area of THIS shape at R=1
        # rescale to exactly pi*R^2
        return R * np.sqrt(np.pi / area) * np.exp(g(phi))       
    

    # get boundaries
    r_cell = boundary(radius, rough)
    r_nuc  = boundary(radius * nuc_frac, rough * 0.6, extra=1.5)

    # check if inside cell / bucleus
    cell = rho <= r_cell
    nuc  = rho <= np.minimum(r_nuc, r_cell - 1.5)     # leave a cytoplasmic rim

    # depth: -1 nucleus centre, 0 envelope, +1 membrane
    tau = np.where(rho < r_nuc, rho/r_nuc - 1.0,
                   (rho - r_nuc) / np.maximum(r_cell - r_nuc, 1e-6))
    
    return cell, nuc, rho, phi, tau

cell, nuc, rho, phi, tau = generate_single_cell()

# plt.imshow(rho)
# plt.show()
# plt.imshow(phi)
# plt.show()
# plt.imshow(cell)
# plt.show()
# plt.imshow(nuc)
# plt.show()
    
    

In [ ]:
def plot_interactive_cell(size, radius, nuc_frac, rough, K, elong, angle_deg, beta):
    # Generate the masks
    cell, nuc, rho, phi, tau = generate_single_cell(
        size=size, radius=radius, nuc_frac=nuc_frac, 
        rough=rough, K=K, elong=elong, 
        angle_deg=angle_deg, beta=beta
    )
    
    # Create an RGB image background (black)
    img = np.zeros((size, size, 3))
    
    # Assign colors using the masks
    # Cytoplasm (light green)
    img[cell] = [0.2, 0.8, 0.3]
    # Nucleus (light blue) overwrites the cytoplasm where it exists
    img[nuc] = [0.3, 0.5, 0.9]
    
    # Plotting
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Synthetic Cell Generation')
    plt.show()

# Set up the interactive sliders with reasonable bounds based on your defaults
interact(plot_interactive_cell,
         size=widgets.IntSlider(min=100, max=500, step=10, value=201, description='Size'),
         radius=widgets.FloatSlider(min=10.0, max=150.0, step=1.0, value=32.0, description='Radius'),
         nuc_frac=widgets.FloatSlider(min=0.1, max=0.9, step=0.05, value=0.3, description='Nuc Frac'),
         rough=widgets.FloatSlider(min=0.0, max=2.0, step=0.1, value=0.9, description='Roughness'),
         K=widgets.IntSlider(min=1, max=50, step=1, value=20, description='K (Harmonics)'),
         elong=widgets.FloatSlider(min=0.5, max=4.0, step=0.1, value=1.6, description='Elongation'),
         angle_deg=widgets.FloatSlider(min=0.0, max=360.0, step=5.0, value=30.0, description='Angle (deg)'),
         beta=widgets.FloatSlider(min=0.5, max=4.0, step=0.1, value=1.9, description='Beta'));

## 1.2) Cell Marker expression

In [ ]:
cell, nuc, rho, phi, tau = generate_single_cell()
fig, ax = plt.subplots()
im = ax.imshow(tau, cmap=plt.get_cmap('RdGy'), interpolation='nearest',
               vmin=-1, vmax=1)
fig.colorbar(im)
plt.show()

In [ ]:
def localization_function(x, mu=.0, width=0.5, sharp=4.0, floor=0.0):
    """ Where the marker is expressed as a function of tau, with a peak at mu, width, sharpness and floor level. """
    f = np.exp(-np.abs((x - mu) / width) ** sharp)
    return ((1.0 - floor) * f + floor)
     
     
def texture(shape, phi, strength=0.5, scale_px=3.0, polarity=0.0, pol_dir=0.0, seed=0):
    """Blobby patchiness × angular polarity."""
    rng = np.random.default_rng(seed)
    z = gaussian_filter(rng.standard_normal(shape), scale_px)
    z /= z.std() + 1e-12
    return np.exp(strength * z + polarity * np.cos(phi - pol_dir))



def render(cell, *factors, amp=1.0):
    """Multiply the shapes, gate by the mask, and set the amount -- normalise ONCE, last."""
    f = cell.astype(float)
    for x in factors:
        f = f * x
    return amp * f / f[cell].mean()

def marker(tau, cell, mu=1.0, width=0.25, sharp=2.0, floor=0.0, amp=1.0):
    f = localization_function(tau, mu=mu, width=width, sharp=sharp, floor=floor)
    texture(shape, phi, strength=0.5, scale_px=3.0, polarity=0.0, pol_dir=0.0, seed=0)
    f = f * cell
    # normalise so the in-cell mean is exactly amp
    return amp * f / f[cell].mean()

In [ ]:
# --- Interactive Plotting Function ---
def plot_combined_interactive(cell, tau, phi, 
                              mu, width, sharp, floor, amp, 
                              strength, scale_px, polarity, pol_dir):
    seed = 42
    
    # 1. Compute 1D Line Data
    x = np.linspace(-1, 1, 201)
    y_1d = localization_function(x, mu=mu, width=width, sharp=sharp, floor=floor)
    
    # 2. Compute 2D Marker Profile Image (using tau)
    y_2d = localization_function(tau, mu=mu, width=width, sharp=sharp, floor=floor)
    
    # 3. Generate Texture with slider parameters
    texture_img = texture(cell.shape, phi, 
                          strength=strength, 
                          scale_px=scale_px, 
                          polarity=polarity, 
                          pol_dir=pol_dir,
                          seed = seed)
    
    # 4. Render final profile
    marker_profile = render(cell, y_2d, texture_img, amp=amp)
    
    # Set up the figure with 1 row and 4 columns
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    ax0, ax1, ax2, ax3 = axes.flatten()
    
    # --- Plot 1: 1D Line Plot ---
    ax0.plot(x, y_1d, color='blue', linewidth=2, label=f'μ={mu}\nw={width}\ns={sharp}')
    ax0.set_title('1D Localization Profile')
    ax0.set_xlabel('tau')
    ax0.set_ylabel('Intensity')
    ax0.set_ylim(bottom=0)
    ax0.grid(True, linestyle='--', alpha=0.7)
    ax0.legend(loc='upper right')
    
    # --- Plot 2: 2D Localization Only ---
    im1 = ax1.imshow(y_2d * cell, vmin=0, vmax=1, cmap='viridis')
    ax1.axis('off')
    ax1.set_title('2D Localization Only')
    fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
    
    # --- Plot 3: Texture Only ---
    im2 = ax2.imshow(texture_img * cell, cmap='viridis')
    ax2.axis('off')
    ax2.set_title('Texture Pattern Only')
    fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
    
    # --- Plot 4: Final Combined Expression ---
    im3 = ax3.imshow(marker_profile, vmin=0, vmax=1, cmap='viridis')
    ax3.axis('off')
    ax3.set_title('Final Rendered Expression')
    fig.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04, label='Expression Level')
    
    plt.tight_layout()
    plt.show()

# (Assumes generate_single_cell is already defined and run in your notebook)
cell, nuc, rho, phi, tau = generate_single_cell()

# --- Initialize Sliders ---
interact(plot_combined_interactive,
         cell=fixed(cell), 
         tau=fixed(tau), 
         phi=fixed(phi),
         # Localization parameters
         mu=widgets.FloatSlider(min=-1.0, max=2.0, step=0.1, value=1.0, description='Mu (Center)'),
         width=widgets.FloatSlider(min=0.05, max=1.5, step=0.05, value=0.25, description='Width'),
         sharp=widgets.FloatSlider(min=0.5, max=10.0, step=0.5, value=2.0, description='Sharpness'),
         floor=widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.0, description='Floor'),
         amp=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='Amplitude'),
         # Texture parameters
         strength=widgets.FloatSlider(min=0.0, max=3.0, step=0.1, value=0.5, description='Tex Strength'),
         scale_px=widgets.FloatSlider(min=0.5, max=15.0, step=0.5, value=3.0, description='Tex Scale (px)'),
         polarity=widgets.FloatSlider(min=-2.0, max=2.0, step=0.1, value=0.0, description='Tex Polarity'),
         pol_dir=widgets.FloatSlider(min=-np.pi, max=np.pi, step=0.1, value=0.0, description='Tex Pol Dir'));

# 2) Synthetic tissue generation